In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from scipy import stats
from sklearn.preprocessing import (MinMaxScaler, PowerTransformer, RobustScaler, StandardScaler)


sns.set_theme(style="whitegrid", palette="colorblind")

In [11]:
nomes_colunas = {
    "tier" : "padrao",
    "rooms" : "quartos",
    "parking spaces" : "vagas_garagem",
    "floor" : "andar",
    "total_floors" : "andares_predio",
    "transit_distance_min" : "distancia_transporte",
    "furniture" : "mobiliado",
    "hoa (R$)" : "condominio",
    "rent amount (R$)" : "aluguel",
    "property tax (R$)" : "iptu",
    "total (R$)" : "custo_do_imovel",
}

In [12]:
imoveis = (
    pd.read_csv(r"..\..\data\imoveis_sao_paulo.csv")
    .rename(columns = nomes_colunas)
    .drop_duplicates()
    .reset_index(drop=True)
    .assign(
       tipo_imovel = lambda df: np.where(
           df["andar"].fillna(0) == 0,
           "casa",
           "apartamento"
       ) 
    )
    .astype({
        "vagas_garagem": "Int64",
        "andar": "Int64",
        "andares_predio": "Int64"
    })
)

In [13]:
imoveis.head()

,bairro,zona,padrao,area,quartos,vagas_garagem,andar,andares_predio,distancia_transporte,mobiliado,condominio,aluguel,iptu,custo_do_imovel,tipo_imovel
0,Santo Amaro,Zona Sul,mid,119.0,3,0,11,15,3,not furnished,NaN,5050.0,590.25,6910.0,apartamento
1,Vila Mariana,Zona Sul,premium,97.9,3,1,15,25,10,not furnished,1072.0,5060.0,833.17,6970.0,apartamento
2,Ipiranga,Zona Sul,mid,45.5,1,0,1,15,27,furnished,520.0,2470.0,283.92,3280.0,apartamento
3,Raposo Tavares,Zona Oeste,mid,180.4,4,1,8,15,94,not furnished,2274.0,6270.0,775.17,9320.0,apartamento
4,Brooklin,Zona Sul,luxury,36.1,0,1,2,8,1,not furnished,570.0,2960.0,330.00,3860.0,apartamento


### Criação do cenário para imputação

In [14]:
# Função de apagamento e gabarito base.
aleatorio = np.random.default_rng(42)

def apagar_aleatoriamente(valores, proporcao):
    # Gera um número aleatório para cada linha. Se for menor que a proporção (ex: 0.2 para 20%), 
    # o método .mask() substitui o valor original por um valor nulo (NaN)
    
    return valores.mask(aleatorio.random(len(valores)) < proporcao)

gabarito_imoveis = (
    imoveis
    .dropna(subset=["condominio", "vagas_garagem"])
    .reset_index(drop=True)
)

# Apaga valores da coluna de tipos diferentes
imoveis_ausentes = (
    gabarito_imoveis
    .assign(
        quartos= lambda df: apagar_aleatoriamente(df["quartos"], .10),
        aluguel= lambda df: apagar_aleatoriamente(df["aluguel"], .15),
        mobiliado= lambda df: apagar_aleatoriamente(df["mobiliado"], .08),
    )
)

In [15]:
(
    imoveis_ausentes[["quartos", "aluguel", "mobiliado"]]
    .isna()
    .sum()
    .to_frame("ausentes")
    .assign(
        proporcao = lambda x: (x.ausentes / len(imoveis_ausentes)).round(3)*100
    )


)

,ausentes,proporcao
quartos,2783,9.7
aluguel,4308,15.0
mobiliado,2254,7.9


### 1. Compare o MAE da média e da mediana para rooms e relacione o resultado à assimetria e ao caráter discreto da variável.

In [21]:
# # Constantes para imputação:

quartos_media = (
    imoveis_ausentes.quartos.mean()
)

quartos_mediana = (
    imoveis_ausentes.quartos.median()
)

In [22]:
print(f'Média dos quartos: {quartos_media:.2f}')
print(f'Médiana dos quartos: {quartos_mediana}')

Média dos quartos: 2.26
Médiana dos quartos: 2.0


A média informa um número fracionado, onde a categoria é de inteiros. 

In [23]:
quartos_ausente = imoveis_ausentes.quartos.isna()

quartos_valor_real = gabarito_imoveis.loc[quartos_ausente, "aluguel"]

In [24]:
# # Imputação por mediana de cada padrao 

quartos_padrao_mediana = (
    imoveis_ausentes
    .groupby("padrao")
    .quartos
    .transform(lambda x: x.fillna(x.median()))
    .fillna(quartos_mediana)
)

quartos_padrao_media = (
    imoveis_ausentes
    .groupby("padrao")
    .quartos
    .transform(lambda x: x.fillna(x.mean()))
    .fillna(quartos_media)
)

In [25]:
# MAE de cada estratégia, apenas nas posições imputadas
(
    pd.Series(
        {
            "Média": (
                (quartos_valor_real - quartos_media)
                .abs()
                .mean()
            ),
            "Mediana": (
                (quartos_valor_real - quartos_mediana)
                .abs()
                .mean()
            ),
            "Mediana por padrões": (
                (quartos_valor_real - quartos_padrao_mediana[quartos_ausente])
                .abs()
                .mean()
            ),

            "Media por padrões": (
                (quartos_valor_real - quartos_padrao_media[quartos_ausente])
                .abs()
                .mean()
            ),


        },
        name="MAE (R$)",
    )
    .round(2)
    .to_frame()
)

,MAE (R$)
Média,4005.86
Mediana,4006.12
Mediana por padrões,4005.99
Media por padrões,4005.86
